# 3.6 Analysis of the Spikes in HTTPS Resource Record Adoption


In [ ]:
import pandas as pd
import json
from collections import Counter

apex_before_path = "../data/parsed/2024-10/2024-10-01/apex_https.csv"
apex_after_path  = "../data/parsed/2025-01/2025-01-01/apex_https.csv"
www_before_path  = "../data/parsed/2024-10/2024-10-01/www_https.csv"
www_after_path   = "../data/parsed/2025-01/2025-01-01/www_https.csv"


def safe_json_load(s):
    """Parse JSON-like strings safely; return {} on failure/empty."""
    if pd.isna(s):
        return {}
    s = str(s).strip()
    if not s or s == "{}":
        return {}
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(json.loads(s))
        except Exception:
            return {}


def base_domain(name, labels=2):
    """
    Extract a provider-like base from a hostname.
    Example: 'ns1.google.com.' -> 'google.com'
             'youtube-ui.l.google.com.' -> 'google.com'
    """
    if not isinstance(name, str):
        return None
    name = name.strip().rstrip(".")
    if not name:
        return None
    parts = name.split(".")
    if len(parts) <= labels:
        return name
    return ".".join(parts[-labels:])


def ns_provider_field(ns_field):
    """
    Extract an NS provider base domain from the JSON-encoded 'ns' field.
    """
    data = safe_json_load(ns_field)
    ns_list = []
    if isinstance(data, dict):
        ns_list = data.get("NS") or data.get("ns") or []
    elif isinstance(data, list):
        ns_list = data
    if not ns_list:
        return None
    return base_domain(ns_list[0])


def summarize_dataset(before_path, after_path, label):
    """
    For either APEX or WWW:
      - count HTTPS domains before/after,
      - find domains only in AFTER (new HTTPS domains / new Tranco members),
      - show NS provider distribution changes,
      - show NS provider changes for overlapping domains.
    """
    print("\n" + "=" * 70)
    print(f"DATASET: {label}")
    print("=" * 70)

    # Load CSVs
    before = pd.read_csv(before_path)
    after = pd.read_csv(after_path)

    print(f"Number of HTTPS domains BEFORE: {len(before)}")
    print(f"Number of HTTPS domains AFTER : {len(after)}")
    print(f"Change in HTTPS domains       : {len(after) - len(before)}")
    print()

    # Domain sets
    before_domains = set(before["domain"])
    after_domains = set(after["domain"])

    overlap_domains = before_domains & after_domains
    new_domains = after_domains - before_domains
    lost_domains = before_domains - after_domains

    print(f"Overlapping domains (HTTPS in both snapshots): {len(overlap_domains)}")
    print(f"Domains only BEFORE (HTTPS then gone)       : {len(lost_domains)}")
    print(f"Domains only AFTER  (new HTTPS / new rank)  : {len(new_domains)}")
    print()

    # Compute NS provider for each row
    for df in (before, after):
        df["ns_provider"] = df["ns"].apply(ns_provider_field)

    # ---- Provider growth across all HTTPS domains ---- #
    before_counts = Counter(before["ns_provider"].dropna())
    after_counts  = Counter(after["ns_provider"].dropna())

    all_providers = sorted(set(before_counts) | set(after_counts))
    rows = []
    for p in all_providers:
        rows.append({
            "provider": p,
            "count_before": before_counts.get(p, 0),
            "count_after": after_counts.get(p, 0),
            "delta": after_counts.get(p, 0) - before_counts.get(p, 0)
        })
    prov_df = pd.DataFrame(rows).sort_values("delta", ascending=False)

    print("Top NS providers by growth in # of HTTPS domains (all domains):")
    if not prov_df.empty:
        print(prov_df.head(10).to_string(index=False))
    else:
        print("  (no NS provider information available)")
    print()

    # ---- Providers among domains that appear only AFTER ---- #
    if new_domains:
        after_new = after[after["domain"].isin(new_domains)].copy()
        new_counts = Counter(after_new["ns_provider"].dropna())
        total_new = len(after_new)

        print("NS providers among domains that appear only AFTER (new HTTPS domains / new in Tranco):")
        for provider, count in new_counts.most_common(10):
            share = count / total_new
            print(f"  {provider:30s} {count:6d} ({share:.2%})")
    else:
        print("No domains appear only in AFTER snapshot.")
    print()

    # ---- NS provider changes for overlapping domains ---- #
    if overlap_domains:
        before_overlap = before[before["domain"].isin(overlap_domains)][["domain", "ns_provider"]]
        after_overlap  = after[after["domain"].isin(overlap_domains)][["domain", "ns_provider"]]

        overlap_df = before_overlap.merge(
            after_overlap,
            on="domain",
            suffixes=("_before", "_after")
        )
        overlap_df["changed"] = overlap_df["ns_provider_before"] != overlap_df["ns_provider_after"]
        changed = overlap_df[overlap_df["changed"]]

        print(f"Overlapping domains with NS provider change: {len(changed)}")
        if not changed.empty:
            change_pairs = Counter(zip(changed["ns_provider_before"], changed["ns_provider_after"]))
            print("Most common NS provider change pairs:")
            for (b, a), c in change_pairs.most_common(10):
                print(f"  {str(b):20s} -> {str(a):20s} : {c}")
        else:
            print("No NS provider changes among overlapping domains.")
    else:
        print("No overlapping domains to analyze for NS changes.")

    print("\n" + "-" * 70)


summarize_dataset(apex_before_path, apex_after_path, label="APEX")
summarize_dataset(www_before_path,  www_after_path,  label="WWW")


DATASET: APEX
Number of HTTPS domains BEFORE: 239802
Number of HTTPS domains AFTER : 269379
Change in HTTPS domains       : 29577

Overlapping domains (HTTPS in both snapshots): 172391
Domains only BEFORE (HTTPS then gone)       : 67411
Domains only AFTER  (new HTTPS / new rank)  : 96988

Top NS providers by growth in # of HTTPS domains (all domains):
              provider  count_before  count_after  delta
          facebook.com             0          112    112
     foundationdns.com            11           23     12
        cloudflare.net            48           56      8
          neodigit.net             2            9      7
                 co.uk             4           11      7
     foundationdns.org            20           27      7
        informadns.com             8           14      6
globaldomainserver.net            87           92      5
              icsn.com            15           19      4
            flexbe.com             0            3      3

NS providers amon

# Section 3.7: Analysis of the Dips in DNSSEC Coverage


In [ ]:
import pandas as pd
import json
from collections import Counter

apex_before_path = "../data/parsed/2024-10/2024-10-01/apex_https.csv"
apex_after_path  = "../data/parsed/2025-01/2025-01-01/apex_https.csv"
www_before_path  = "../data/parsed/2024-10/2024-10-01/www_https.csv"
www_after_path   = "../data/parsed/2025-01/2025-01-01/www_https.csv"


def safe_json_load(s):
    """Parse JSON-like strings safely; return {} on failure or empty."""
    if pd.isna(s):
        return {}
    s = str(s).strip()
    if not s or s == "{}":
        return {}
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(json.loads(s))
        except Exception:
            return {}

def extract_rrsig_present(https_field):
    """Return 1 if HTTPS JSON has a non-empty RRSIG list, else 0."""
    data = safe_json_load(https_field)
    if not isinstance(data, dict):
        return 0
    rrsig = data.get("RRSIG")
    return 1 if isinstance(rrsig, list) and len(rrsig) > 0 else 0

def extract_adbit(https_field):
    """Return 1 if AD bit is recorded as true in HTTPS JSON, else 0."""
    data = safe_json_load(https_field)
    return 1 if data.get("AD", False) else 0

def base_domain(name, labels=2):
    """Extract provider-like base domain from a hostname."""
    if not isinstance(name, str):
        return None
    name = name.strip().rstrip(".")
    if not name:
        return None
    parts = name.split(".")
    if len(parts) <= labels:
        return name
    return ".".join(parts[-labels:])

def ns_provider_field(ns_field):
    """Extract NS provider base domain from JSON-encoded ns field."""
    data = safe_json_load(ns_field)
    ns_list = []
    if isinstance(data, dict):
        ns_list = data.get("NS", []) or data.get("ns", [])
    elif isinstance(data, list):
        ns_list = data
    if not ns_list:
        return None
    return base_domain(ns_list[0])


def analyze_dataset(before_path, after_path, label):
    print("\n" + "="*70)
    print(f"DATASET: {label}")
    print("="*70)

    # Load
    before = pd.read_csv(before_path)
    after  = pd.read_csv(after_path)

    # Compute DNSSEC indicators
    for df in (before, after):
        df["rrsig_present"] = df["https"].apply(extract_rrsig_present)
        df["ad_present"]    = df["https"].apply(extract_adbit)
        df["ns_provider"]   = df["ns"].apply(ns_provider_field)

    total_before = len(before)
    total_after  = len(after)

    frac_rrsig_before = before["rrsig_present"].mean()
    frac_rrsig_after  = after["rrsig_present"].mean()

    frac_ad_before = before["ad_present"].mean()
    frac_ad_after  = after["ad_present"].mean()

    print("Dynamic (all domains in each snapshot)")
    print("--------------------------------------")
    print(f"Total domains BEFORE: {total_before}")
    print(f"Total domains AFTER : {total_after}")
    print(f"RRSIG coverage BEFORE: {frac_rrsig_before:.4f}")
    print(f"RRSIG coverage AFTER : {frac_rrsig_after:.4f}")
    print(f"AD-bit coverage BEFORE: {frac_ad_before:.4f}")
    print(f"AD-bit coverage AFTER : {frac_ad_after:.4f}")
    print()

    overlap_domains = set(before["domain"]) & set(after["domain"])
    before_ov = before[before["domain"].isin(overlap_domains)].copy()
    after_ov  = after[after["domain"].isin(overlap_domains)].copy()

    frac_rrsig_before_ov = before_ov["rrsig_present"].mean()
    frac_rrsig_after_ov  = after_ov["rrsig_present"].mean()

    frac_ad_before_ov = before_ov["ad_present"].mean()
    frac_ad_after_ov  = after_ov["ad_present"].mean()

    print("Overlapping domains (present in both snapshots)")
    print("-----------------------------------------------")
    print(f"Overlapping domains: {len(overlap_domains)}")
    print(f"RRSIG coverage BEFORE (overlap): {frac_rrsig_before_ov:.4f}")
    print(f"RRSIG coverage AFTER  (overlap): {frac_rrsig_after_ov:.4f}")
    print(f"AD-bit coverage BEFORE (overlap): {frac_ad_before_ov:.4f}")
    print(f"AD-bit coverage AFTER  (overlap): {frac_ad_after_ov:.4f}")
    print()

    merged = before_ov[["domain","rrsig_present","ad_present","ns_provider"]].merge(
        after_ov[["domain","rrsig_present","ad_present","ns_provider"]],
        on="domain",
        suffixes=("_before","_after")
    )

    lost_rrsig = merged[(merged["rrsig_present_before"] == 1) &
                        (merged["rrsig_present_after"] == 0)]
    gained_rrsig = merged[(merged["rrsig_present_before"] == 0) &
                          (merged["rrsig_present_after"] == 1)]

    lost_ad = merged[(merged["ad_present_before"] == 1) &
                     (merged["ad_present_after"] == 0)]
    gained_ad = merged[(merged["ad_present_before"] == 0) &
                       (merged["ad_present_after"] == 1)]

    print("RRSIG and AD-bit changes for overlapping domains")
    print("-----------------------------------------------")
    print(f"Domains that lost RRSIG: {len(lost_rrsig)}")
    print(f"Domains that gained RRSIG: {len(gained_rrsig)}")
    print(f"Domains that lost AD-bit: {len(lost_ad)}")
    print(f"Domains that gained AD-bit: {len(gained_ad)}")
    print()

    if len(lost_rrsig) > 0:
        print("Top NS providers among domains that LOST RRSIG:")
        ctr_lost_rrsig = Counter(lost_rrsig["ns_provider_before"].dropna())
        for prov, cnt in ctr_lost_rrsig.most_common(10):
            print(f"  {prov:30s} {cnt}")
        print()
    else:
        print("No domains lost RRSIG")
        print()

    if len(gained_rrsig) > 0:
        print("Top NS providers among domains that GAINED RRSIG:")
        ctr_gain_rrsig = Counter(gained_rrsig["ns_provider_after"].dropna())
        for prov, cnt in ctr_gain_rrsig.most_common(10):
            print(f"  {prov:30s} {cnt}")
        print()
    else:
        print("No domains gained RRSIG")
        print()

    if len(lost_ad) > 0:
        print("Top NS providers among domains that LOST AD-bit:")
        ctr_lost_ad = Counter(lost_ad["ns_provider_before"].dropna())
        for prov, cnt in ctr_lost_ad.most_common(10):
            print(f"  {prov:30s} {cnt}")
        print()
    else:
        print("No domains lost AD-bit")
        print()

    if len(gained_ad) > 0:
        print("Top NS providers among domains that GAINED AD-bit:")
        ctr_gain_ad = Counter(gained_ad["ns_provider_after"].dropna())
        for prov, cnt in ctr_gain_ad.most_common(10):
            print(f"  {prov:30s} {cnt}")
        print()
    else:
        print("No domains gained AD-bit")
        print()

    merged["ns_changed"] = merged["ns_provider_before"] != merged["ns_provider_after"]
    changed = merged[merged["ns_changed"]]

    print("NS provider transitions among overlapping domains")
    print("-------------------------------------------------")
    print(f"Domains with NS provider change: {len(changed)}")
    if len(changed) > 0:
        pairs = Counter(zip(changed["ns_provider_before"], changed["ns_provider_after"]))
        print("Most common NS provider change pairs:")
        for (before_ns, after_ns), count in pairs.most_common(10):
            print(f"  {str(before_ns):20s} -> {str(after_ns):20s} : {count}")
    print("\n" + "-"*70)


analyze_dataset(apex_before_path, apex_after_path, "APEX")
analyze_dataset(www_before_path,  www_after_path,  "WWW")


DATASET: APEX
Dynamic (all domains in each snapshot)
--------------------------------------
Total domains BEFORE: 239802
Total domains AFTER : 269379
RRSIG coverage BEFORE: 0.0808
RRSIG coverage AFTER : 0.0746
AD-bit coverage BEFORE: 0.0000
AD-bit coverage AFTER : 0.0000

Overlapping domains (present in both snapshots)
-----------------------------------------------
Overlapping domains: 172391
RRSIG coverage BEFORE (overlap): 0.0974
RRSIG coverage AFTER  (overlap): 0.0997
AD-bit coverage BEFORE (overlap): 0.0000
AD-bit coverage AFTER  (overlap): 0.0000

RRSIG and AD-bit changes for overlapping domains
-----------------------------------------------
Domains that lost RRSIG: 75
Domains that gained RRSIG: 466
Domains that lost AD-bit: 0
Domains that gained AD-bit: 0

Top NS providers among domains that LOST RRSIG:
  cloudflare.com                 73
  mls.nc                         1
  cloudflare.net                 1

Top NS providers among domains that GAINED RRSIG:
  cloudflare.com   

# 3.5.1 NS Providers and DNSSEC Coverage for ECH-Enabled Domains


In [2]:
import pandas as pd
from pathlib import Path

base = Path("../data/plotting/alldom")

# load the daily aggregates we just regenerated
apex = pd.read_csv(base / "ech_apex.csv")
www = pd.read_csv(base / "ech_www.csv")

def summarize_ns_providers(df, label):
    print(f"\n=== {label} ===")
    # how many ECH days are all-Cloudflare, mixed, etc.
    counts = (
        df[["num_ech", "num_ech_cf_only", "num_ech_non_cloudflare",
            "num_ech_ns_mixed", "num_ech_ns_unknown"]]
        .sum()
        .rename({
            "num_ech": "total_ech",
            "num_ech_cf_only": "cloudflare_only",
            "num_ech_non_cloudflare": "non_cloudflare_only",
            "num_ech_ns_mixed": "cloudflare+other",
            "num_ech_ns_unknown": "unknown_ns",
        })
    )
    print(counts.to_frame("domains"))

    # list any alternate providers that appear on the same day as ECH
    other = (
        df.loc[df["ech_ns_non_cf_providers"] != "none",
               "ech_ns_non_cf_providers"]
        .str.split(";")
        .explode()
        .value_counts()
        .rename_axis("provider")
        .rename("days_with_ech")
    )
    if other.empty:
        print("Only Cloudflare shows up with ECH.")
    else:
        print("\nNon-Cloudflare NS present with ECH:")
        display(other)

    # dnssec overlap: count & percentage already precomputed per day
    pct = df["pct_ech_dnssec"].dropna()
    if pct.empty:
        print("No DNSSEC data for ECH domains.")
    else:
        print(
            f"\nECH domains with DNSSEC: "
            f"{df['num_ech_dnssec'].sum()} of {df['num_ech'].sum()} "
            f"({pct.mean():.2f}% daily average)"
        )

summarize_ns_providers(apex, "APEX")
summarize_ns_providers(www, "WWW")


=== APEX ===
                      domains
total_ech            49178888
cloudflare_only      49102741
non_cloudflare_only     75920
cloudflare+other           22
unknown_ns                205

Non-Cloudflare NS present with ECH:


provider
domainactive.org       323
salla.cloud            321
co.id                  320
bitcoinmagazine.com    310
ubmdns.com             310
                      ... 
rumahweb.net             1
rumahweb.com             1
gob.pa                   1
contabo.net              1
mtweek.com               1
Name: days_with_ech, Length: 226, dtype: int64


ECH domains with DNSSEC: 2886379 of 49178888 (33.09% daily average)

=== WWW ===
                      domains
total_ech            45892219
cloudflare_only        186709
non_cloudflare_only      2726
cloudflare+other            0
unknown_ns           45702784

Non-Cloudflare NS present with ECH:


provider
registrar.eu              283
gorbe.ninja               170
hennepinhealthcare.org    152
selzy.com                 144
foundationdns.org         124
                         ... 
solidhosting.pro            1
fkzf.com                    1
hostdl.com                  1
awsdns-23.org               1
inhostedns.org              1
Name: days_with_ech, Length: 72, dtype: int64


ECH domains with DNSSEC: 2716939 of 45892219 (33.31% daily average)


# 3.5.2 One-Day Snapshot Analysis of HTTPS Resource Record Parameters


In [3]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from pathlib import Path
import sys

project_root = Path("..").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import preprocessfns as fns
# --- configuration for the one-day snapshot ---
snapshot_date = "2025-07-01"   # YYYY-MM-DD inside data/parsed/<YYYY-MM>/<YYYY-MM-DD>/
dom_type = "apex"              # "apex" or "www"
data_root = Path("../data/parsed")
csv_path = data_root / snapshot_date[:7] / snapshot_date / f"{dom_type}_https.csv"
print(f"Loading {csv_path}")

# --- helpers ---------------------------------------------------------------

def safe_json(value):
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text and text != "{}":
            try:
                return json.loads(text)
            except Exception:
                pass
    return {}

def base_domain(name: str, labels: int = 2):
    if not isinstance(name, str):
        return None
    stripped = name.strip().rstrip(".")
    if not stripped:
        return None
    parts = stripped.split(".")
    if len(parts) <= labels:
        return stripped.lower()
    return ".".join(parts[-labels:]).lower()

def extract_ns_providers(ns_field):
    data = safe_json(ns_field)
    if isinstance(data, dict):
        candidates = data.get("NS") or data.get("ns") or []
    elif isinstance(data, list):
        candidates = data
    else:
        candidates = []
    providers = {
        base_domain(name) for name in candidates if base_domain(name)
    }
    return tuple(sorted(providers))

def to_list(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        return [item.strip() for item in value.split(",") if item.strip()]
    return [value]

def classify_target(row):
    target = row.get("TargetName")
    if target is None or (isinstance(target, float) and np.isnan(target)):
        return "missing"
    target = str(target).strip()
    if target == ".":
        return "dot"
    domain = str(row.get("domain", "")).strip().rstrip(".")
    target_clean = target.rstrip(".")
    if target_clean.lower() == domain.lower():
        return "same_as_domain"
    return "other_domain"

# --- load + enrich snapshot -----------------------------------------------

df = pd.read_csv(csv_path)
print(f"rows: {len(df)}")

df["https.dict"] = df["https"].apply(safe_json)
df = df[df["https.dict"].apply(lambda x: isinstance(x, dict) and fns.if_key_exists(x, "HTTPS"))].copy()
print(f"HTTPS RR present: {len(df)}")

df["SvcPriority"] = df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "SvcPriority"))
df["TargetName"] = df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "TargetName"))
df["alpn"] = df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "alpn"))
df["ipv4hint"] = df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "svcb.ipv4hint"))
df["ipv6hint"] = df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "svcb.ipv6hint"))
df["ipv4hint_count"] = df["ipv4hint"].apply(lambda v: len(set(to_list(v))))
df["ipv6hint_count"] = df["ipv6hint"].apply(lambda v: len(set(to_list(v))))
df["TargetCategory"] = df.apply(classify_target, axis=1)

df["ns.providers"] = df["ns"].apply(extract_ns_providers)
df["primary_ns_provider"] = df["ns.providers"].apply(lambda providers: providers[0] if providers else "unknown")

# RRSIG (DNSSEC) overlap for context
df["has_rrsig"] = df["https.dict"].apply(lambda x: bool(fns.if_key_exists(x, "RRSIG")))

# --- outputs --------------------------------------------------------------

print("\nTop NS providers for HTTPS (primary label):")
display(df["primary_ns_provider"].value_counts().head(15))

print("\nSvcPriority distribution:")
display(df["SvcPriority"].value_counts(dropna=False))

print("\nTargetName classification (., same as domain, other):")
display(df["TargetCategory"].value_counts())

print("\nalpn parameter (as-is):")
display(df["alpn"].value_counts(dropna=False).head(15))

print("\nDistinct IP counts per record:")
ipv_counts = df[["ipv4hint_count", "ipv6hint_count"]].describe().T[["count", "mean", "min", "max"]]
display(ipv_counts)

print("\nHow many HTTPS records list any non-Cloudflare NS?")
non_cf = df[df["ns.providers"].apply(lambda providers: any("cloudflare" not in p for p in providers))]
print(f"{len(non_cf)} / {len(df)} ({len(non_cf) / max(len(df),1):.2%}) records have at least one non-Cloudflare NS provider")

print("\nECH + DNSSEC overlap (for reference):")
ech = df[df["https.dict"].apply(lambda x: fns.get_https_fields(x, "HTTPS", "svcb.ech") is not None)]
if ech.empty:
    print("No ECH-enabled HTTPS records in this snapshot.")
else:
    pct_dnssec = ech["has_rrsig"].mean() * 100
    print(f"{ech['has_rrsig'].sum()} of {len(ech)} ECH records carry RRSIG ({pct_dnssec:.2f}%)")

Loading ../data/parsed/2025-07/2025-07-01/apex_https.csv


/var/folders/87/_54hxc9j7ml9lrvh4z52b3rm0000gn/T/ipykernel_49514/434369374.py:86: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


rows: 263861
HTTPS RR present: 223193

Top NS providers for HTTPS (primary label):


primary_ns_provider
cloudflare.com            221253
facebook.com                 336
foundationdns.com            175
google.com                   155
globaldomainserver.net        87
domaincontrol.com             59
salla.cloud                   57
ubmdns.com                    57
cloudflare.net                51
icsn.com                      39
domainactive.org              37
ibm.com                       15
jh-cf.com                     15
googledomains.com             14
hyp.net                       12
Name: count, dtype: int64


SvcPriority distribution:


...


alpn parameter (as-is):


alpn
None    223193
Name: count, dtype: int64


Distinct IP counts per record:


,count,mean,min,max
ipv4hint_count,223193.0,3.059048,0.0,8.0
ipv6hint_count,223193.0,2.964878,0.0,8.0



How many HTTPS records list any non-Cloudflare NS?
1893 / 223193 (0.85%) records have at least one non-Cloudflare NS provider

ECH + DNSSEC overlap (for reference):
9989 of 148405 ECH records carry RRSIG (6.73%)
